# Quiz Practice — Random Forest vs LightGBM

**Goal:** practice the standard workflow for evaluating ensemble classifiers on the breast cancer dataset.

We'll build **two models** and compare them:
1. `RandomForestClassifier` (bagging)
2. `LGBMClassifier` (gradient boosting — LightGBM)

For each: split → fit → predict → evaluate with confusion matrix, classification report, ROC-AUC.

This is about **learning the workflow**, not about getting any specific number — focus on what each step does and why.

## 1 — Imports and warning suppression

We import everything upfront and silence noisy LightGBM warnings. The warnings about feature names are harmless — LightGBM just complains when feature names aren't passed as a DataFrame. Suppressing keeps the output readable.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

# LightGBM — install with `pip install lightgbm` if missing
from lightgbm import LGBMClassifier

## 2 — Load the breast cancer dataset

`load_breast_cancer()` returns a sklearn Bunch (dict-like object) with:
- `.data` → feature matrix X (569 rows, 30 numerical features)
- `.target` → labels y (0 = malignant, 1 = benign)
- `.feature_names`, `.target_names` → metadata

Binary classification. ~63% benign / ~37% malignant — slightly imbalanced, which is why we'll use `class_weight='balanced'` later.

In [ ]:
data = load_breast_cancer()
X = data.data
y = data.target

print('Shape of X:', X.shape)
print('Classes:', data.target_names)
print('Class distribution:', np.bincount(y))   # [malignant, benign]
print(f'Imbalance ratio: {np.bincount(y)[1] / np.bincount(y)[0]:.2f}')

## 3 — Train / test split

Three key choices here:

- **`test_size=0.2`** — 80:20 split. 20% held out for final evaluation.
- **`random_state=9001`** — fixes the split so results are reproducible.
- **`stratify=y`** — preserves class proportions in both train and test.

Without `stratify`, a random split might accidentally put more malignant cases in one set than the other, biasing your evaluation. Stratification guarantees both sets have the same class ratio as the full dataset.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=9001,
    stratify=y
)

print('Train:', X_train.shape, ' Class dist:', np.bincount(y_train))
print('Test :', X_test.shape,  ' Class dist:', np.bincount(y_test))

## 4 — Build the Random Forest model

Hyperparameters explained:

- **`class_weight='balanced'`** — automatically weights samples inversely to class frequency. Helps with the slight class imbalance.
- **`random_state=9001`** — same RNG seed so trees and bootstrap samples are reproducible.
- **`n_estimators=100`** — train 100 trees. Standard starting point — more isn't always better.
- **`max_depth=4`** — limit each tree to depth 4. Unusually shallow for RF (usually let trees grow deep), but it shows RF can work even with restricted trees.

Recall: Random Forest = bagging (bootstrap samples) + random feature subsets per split.

In [ ]:
rf = RandomForestClassifier(
    class_weight='balanced',
    random_state=9001,
    n_estimators=100,
    max_depth=4
)

## 5 — Fit Random Forest on training data

`fit()` trains all 100 trees in parallel — each on its own bootstrap sample, each using random feature subsets at every split. After this call, the model is ready to predict.

In [ ]:
rf.fit(X_train, y_train)
print('Random Forest fitted ✓')

## 6 — Predict on test set

Two kinds of predictions:
- `predict()` → class labels (0 or 1) — used for accuracy, confusion matrix
- `predict_proba()` → probabilities — used for ROC-AUC and threshold tuning

For ROC-AUC we need `predict_proba()[:, 1]` — probability of the positive class.

In [ ]:
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

## 7 — Confusion matrix

sklearn's layout: rows = actual, columns = predicted, negative class first.

```
                Pred 0    Pred 1
Actual 0    [   TN        FP   ]
Actual 1    [   FN        TP   ]
```

- **TN** (top-left): correctly predicted malignant
- **FP** (top-right): false alarm — said benign but actually malignant
- **FN** (bottom-left): missed case — said malignant but actually benign
- **TP** (bottom-right): correctly predicted benign

In [ ]:
print('Confusion Matrix — Random Forest:')
print(confusion_matrix(y_test, y_pred_rf))

## 8 — Classification report

Prints precision, recall, F1-score for **each class** plus overall accuracy.

- **precision** = TP / (TP + FP) — of predicted positives, how many real?
- **recall** = TP / (TP + FN) — of actual positives, how many caught?
- **f1** = harmonic mean of precision and recall
- **support** = number of true samples for each class

For binary classification, look at both classes — they tell different stories.

In [ ]:
print('Classification Report — Random Forest:')
print(classification_report(y_test, y_pred_rf, target_names=data.target_names))

## 9 — ROC-AUC score

ROC-AUC measures **discrimination ability across all thresholds**, not just 0.5.

- 1.0 → perfect classifier
- 0.5 → no better than random
- < 0.5 → worse than random (predictions are inverted)

Note that ROC-AUC uses **probabilities** (`y_prob_rf`), not class labels (`y_pred_rf`).

In [ ]:
auc_rf = roc_auc_score(y_test, y_prob_rf)
print(f'ROC-AUC — Random Forest: {auc_rf:.4f}')

---

# Part 2 — LightGBM

LightGBM is a gradient boosting library (alternative to XGBoost / CatBoost). It's:
- **Faster** than sklearn's `GradientBoostingClassifier`
- **Memory-efficient** thanks to histogram-based binning
- **Leaf-wise growth** — grows the leaf that reduces loss most (sklearn grows level-wise)

Install if missing:
```
pip install lightgbm
# or
conda install -c conda-forge lightgbm
```

## 10 — Build the LightGBM model

Hyperparameters explained:

- **`class_weight='balanced'`** — same logic as RF: handle slight imbalance.
- **`random_state=9001`** — reproducibility.
- **`verbose=-1`** — silence training output (no per-iteration logging).
- **`n_estimators=100`** — 100 boosting rounds (sequential trees).
- **`max_depth=2`** — VERY shallow trees. Boosting prefers weak learners — even depth 2 is enough because many trees combine sequentially.
- **`learning_rate=0.1`** — each tree contributes 10% to the ensemble. Smaller = needs more trees but often generalises better.

Note the philosophical difference from RF:
- RF: many DEEP trees in parallel
- LGBM: many SHALLOW trees sequentially

In [ ]:
lgbm = LGBMClassifier(
    class_weight='balanced',
    random_state=9001,
    verbose=-1,
    n_estimators=100,
    max_depth=2,
    learning_rate=0.1
)

## 11 — Fit LightGBM on training data

Unlike RF, this trains sequentially — Tree 1 first, then Tree 2 to fix Tree 1's errors, then Tree 3 to fix the combined errors of Tree 1+2, etc.

In [ ]:
lgbm.fit(X_train, y_train)
print('LightGBM fitted ✓')

## 12 — Predict on test set

Same API as any sklearn classifier — `predict()` and `predict_proba()`.

In [ ]:
y_pred_lgbm = lgbm.predict(X_test)
y_prob_lgbm = lgbm.predict_proba(X_test)[:, 1]

## 13 — Confusion matrix

In [ ]:
print('Confusion Matrix — LightGBM:')
print(confusion_matrix(y_test, y_pred_lgbm))

## 14 — Classification report

In [ ]:
print('Classification Report — LightGBM:')
print(classification_report(y_test, y_pred_lgbm, target_names=data.target_names))

## 15 — ROC-AUC score

In [ ]:
auc_lgbm = roc_auc_score(y_test, y_prob_lgbm)
print(f'ROC-AUC — LightGBM: {auc_lgbm:.4f}')

---

## 16 — Side-by-side comparison

Pull both ROC-AUC scores into one view. Either model can win on a given run — the exact result depends on hyperparameters, the random seed, and which type of error you're optimising for.

In [ ]:
comparison = pd.DataFrame({
    'Model':    ['Random Forest', 'LightGBM'],
    'ROC-AUC':  [round(auc_rf, 4), round(auc_lgbm, 4)],
    'Approach': ['Bagging (parallel, deep trees)', 'Boosting (sequential, shallow trees)']
})
print(comparison)

## Key Learnings

1. **Same workflow for any sklearn classifier**: split → fit → predict → evaluate.

2. **`stratify=y`** is essential whenever class imbalance might matter — keeps the test set representative.

3. **`class_weight='balanced'`** is a quick fix for imbalanced data — re-weights samples without changing the algorithm.

4. **`predict_proba()` is needed for ROC-AUC** — AUC measures discrimination across thresholds, so it needs probabilities not labels.

5. **RF vs LGBM hyperparameters reflect their philosophies:**
   - RF: `max_depth=4` lets each tree learn substantial structure (still 100 of them)
   - LGBM: `max_depth=2` keeps each tree weak but combines many of them sequentially with `learning_rate=0.1`

6. **Same evaluation toolkit for both** — confusion matrix, classification report, ROC-AUC. The metrics don't care which model produced the predictions.

7. **Reproducibility**: always set `random_state` when training. Same seed → same model → same numbers.

8. **Warnings suppression** is fine for tidy notebooks — but read warnings during development; they often catch real issues.